# **Generelization with OOPs: Logistic Regression and Prediction**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score

In [ ]:
class MLBacktester():
  # Class main method
  def __init__(self,companies,period,interval,lags,trade_cost):
    self.companies=companies
    self.period=period
    self.interval=interval
    self.lags=lags
    self.trade_cost=trade_cost
    # self.model=OneVsRestClassifier(LogisticRegression(C=1e6,max_iter=10000))
    self.models={}
    self.get_data()
  # Class String Representation
  def __repr__(self):
    rep=f"MLBackTester(companies={self.companies},period={self.period},interval={self.interval},trade_cost={self.trade_cost})"
    return rep
  # Get the data from yfinance
  def get_data(self):
    self.data=yf.download(self.companies,period=self.period,interval=self.interval)
    self.df=pd.DataFrame(self.data["Close"])
    self.LR=[]
    for i in self.companies:
      self.LR.append(f"LR {i}")
      self.df[f"LR {i}"]=np.log(self.df[i]/self.df[i].shift(1))
    self.df.dropna(inplace=True)
    return self.df
  # Split the data for backtesting and training
  def split_data(self,start,end):
    self.splitted_data=self.df.iloc[start:end].copy()
    return self.splitted_data
  # Preparing Feature column for training & testing
  def prepare_features(self,df_in):
    self.df_subset=df_in.copy()
    self.feature_clm=[]
    for i in self.companies:
      for j in range(1,self.lags+1):
        col=f"Lag {j} of {i}"
        self.df_subset[col]=self.df_subset[f"LR {i}"].shift(j)
        self.feature_clm.append(col)
    self.df_subset.dropna(inplace=True)
    return self.df_subset

  def fit_model(self,start=None,end=None):
    # Prepare Features
    data_features, feature_name=self.prepare_features(self.splitted_data)
    for i in self.companies:
      # Create new model for each i (stock)
      model=OneVsRestClassifier(LogisticRegression(C=1e6,max_iter=10000))
      company_feature=[c for c in feature_name if i in c]
      model.fit(data_features[company_feature],np.sign(data_features[f"LR {i}"]))
      self.models[i]=model    # add the model in dict

  def test_strategy(self,train_ratio,lags):
    self.lags=lags
    # full_data=self.df.copy()
    split_index=int(len(self.df)*train_ratio)
    split_date=self.df.copy().index[split_index-1]
    train_start=self.df.index[0]
    test_end=self.df.index[-1]
    print(f"Training from {train_start} to {split_date}")
    print(f"Testing from {split_date} to {test_end}")
    predict=self.model.predict(self.df_subset[self.feature_clm])
    self.fit_model(train_start,split_date)
    self.prepare_features(split_date,test_end)
    for i in self.companies:
      predict=self.model.predict(self.df_subset[self.feature_clm])
      self.df_subset[f"Pred {i}"]=predict
      self.df_subset[f"Strategy {i}"]=self.df_subset[f"Pred {i}"]*self.df_subset[f"LR {i}"]
      self.df_subset[f"trade {i}"]=self.df_subset[f"Pred {i}"].diff().fillna(0).abs()
      self.df_subset[f"Strategy {i}"]=self.df_subset[f"Strategy {i}"]-(self.df_subset[f"trade {i}"]*self.trade_cost)
      self.df_subset[f"Creturn {i}"]=self.df_subset[f"return {i}"].cumsum().apply(np.exp)
      self.df_subset[f"Cstrategy {i}"]=self.df_subset[f"strategy {i}"].cumsum().apply(np.exp)
      self.result=self.df_subset
      perf=self.result[f"Cstrategy"].iloc[-1]
      outperf=perf-self.result[f"Creturn {i}"].iloc[-1]
      return round(perf,2), round(outperf,6)

In [ ]:
t=MLBacktester(["HAL.NS","BEL.NS","BDL.NS"],"1mo","5m",5,20)
t.get_data()

/tmp/ipython-input-961127137.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  self.data=yf.download(self.companies,period=self.period,interval=self.interval)
[*********************100%***********************]  3 of 3 completed
/tmp/ipython-input-961127137.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  self.data=yf.download(self.companies,period=self.period,interval=self.interval)
[*********************100%***********************]  3 of 3 completed


Ticker,BDL.NS,BEL.NS,HAL.NS,LR HAL.NS,LR BEL.NS,LR BDL.NS
Datetime,,,,,,
2025-12-24 03:50:00+00:00,1445.000000,401.600006,4433.299805,0.000699,0.001869,0.002078
2025-12-24 03:55:00+00:00,1450.199951,401.750000,4445.200195,0.002681,0.000373,0.003592
2025-12-24 04:00:00+00:00,1448.000000,400.750000,4441.000000,-0.000945,-0.002492,-0.001518
2025-12-24 04:05:00+00:00,1447.000000,401.200012,4440.100098,-0.000203,0.001122,-0.000691
2025-12-24 04:10:00+00:00,1447.000000,401.200012,4433.799805,-0.001420,0.000000,0.000000
...,...,...,...,...,...,...
2026-01-23 09:35:00+00:00,1405.500000,408.950012,4300.100098,-0.000907,-0.000855,-0.002700
2026-01-23 09:40:00+00:00,1405.500000,408.750000,4301.700195,0.000372,-0.000489,0.000000
2026-01-23 09:45:00+00:00,1405.300049,410.250000,4301.399902,-0.000070,0.003663,-0.000142


In [ ]:
t.split_data(20,1200)

Ticker,BDL.NS,BEL.NS,HAL.NS,LR HAL.NS,LR BEL.NS,LR BDL.NS
Datetime,,,,,,
2025-12-24 05:30:00+00:00,1485.500000,401.350006,4454.100098,0.000202,-0.000125,-0.000202
2025-12-24 05:35:00+00:00,1478.000000,401.500000,4453.200195,-0.000202,0.000374,-0.005062
2025-12-24 05:40:00+00:00,1480.900024,401.649994,4452.000000,-0.000270,0.000374,0.001960
2025-12-24 05:45:00+00:00,1480.300049,401.850006,4451.200195,-0.000180,0.000498,-0.000405
2025-12-24 05:50:00+00:00,1478.800049,401.649994,4448.000000,-0.000719,-0.000498,-0.001014
...,...,...,...,...,...,...
2026-01-16 09:50:00+00:00,1518.000000,410.549988,4433.000000,0.001241,0.002317,0.000527
2026-01-16 09:55:00+00:00,1512.800049,410.600006,4428.700195,-0.000970,0.000122,-0.003431
2026-01-19 03:45:00+00:00,1528.000000,414.149994,4482.299805,0.012030,0.008609,0.009997


In [ ]:
t.prepare_features(5)

(Ticker                          BDL.NS      BEL.NS       HAL.NS  LR HAL.NS  \
 Datetime                                                                     
 2025-12-24 05:55:00+00:00  1485.500000  402.299988  4451.299805   0.000742   
 2025-12-24 06:00:00+00:00  1490.599976  403.200012  4456.700195   0.001212   
 2025-12-24 06:05:00+00:00  1494.699951  403.250000  4456.000000  -0.000157   
 2025-12-24 06:10:00+00:00  1487.099976  402.200012  4448.399902  -0.001707   
 2025-12-24 06:15:00+00:00  1491.000000  402.049988  4445.700195  -0.000607   
 ...                                ...         ...          ...        ...   
 2026-01-16 09:50:00+00:00  1518.000000  410.549988  4433.000000   0.001241   
 2026-01-16 09:55:00+00:00  1512.800049  410.600006  4428.700195  -0.000970   
 2026-01-19 03:45:00+00:00  1528.000000  414.149994  4482.299805   0.012030   
 2026-01-19 03:50:00+00:00  1520.300049  414.250000  4471.000000  -0.002524   
 2026-01-19 03:55:00+00:00  1513.300049  412.799988 